## AI Research Trends Analysis Project

This project is about analyzing current AI research trends by fetching OpenAlex API data. I inspected the first query (api url), it returned data for future-dated years, because it was sorted only by newest publication date. So now the query has been refined so that the project focuses on publications from 2020 up to today's date.

The workflow of my project is:
API → raw JSON → selected fields → DataFrame → validation → cleaning

In [ ]:
# A function for fetching AI research works from OpenAlex API. The end date is generated with `datetime` library so that future publication dates are not intentionally requested.

import requests
from datetime import date


def fetch_works(api_url):
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from API: {e}")
        return None

current_date = date.today().isoformat()

api_url = (
    "https://api.openalex.org/works"
        "?search=artificial%20intelligence"
        f"&filter=from_publication_date:2020-01-01,to_publication_date:{current_date}"
        "&per_page=100"
)
works_data = fetch_works(api_url)


In the next cell, I am inspecting what OpenAlex API actually returned. This will help me understand the structure of the data and decide which fields to extract for further analysis.

In [ ]:
# Inspecting the API response

if works_data is not None:
    print(type(works_data))
    print(works_data.keys())
    print(type(works_data["results"]))
    print(len(works_data["results"]))
    print(works_data["results"][0]["title"])
    print(works_data["results"][0]["publication_year"])
    print(works_data["results"][0]["cited_by_count"])
    print(works_data["results"][0]["open_access"])

# The above inspection confirms that OpenAlex returns a dictionary. The actual research works are stored inside the `results` list.


<class 'dict'>
dict_keys(['meta', 'results', 'group_by'])
<class 'list'>
100
Artificial Intelligence in Education: A Review
2020
3963
{'is_oa': True, 'oa_status': 'gold', 'oa_url': 'https://doi.org/10.1109/access.2020.2988510', 'any_repository_has_fulltext': True}


In the next step, I am saving the original API response to a JSON file. This helps me to preserve the raw data before transforming or analyzing it further.

In [38]:
# saving the fetched raw data to a JSON file
import json

def save_json(data, file_path):
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


if works_data is not None:
    save_json(works_data, "data/works_data.json")

The OpenAlex response contains many nested fields.  
For the first version of the project, I have selected only four fields:

- title
- publication year
- citation count
- open-access status

In [ ]:
# Extracting the relevant information from the works data

def extract_relevant_fields(works_data):
    results = works_data["results"]

    extracted_data = []

    for work in results:
        extracted_data.append({
            "title": work["title"],
            "publication_year": work["publication_year"],
            "cited_by_count": work["cited_by_count"],
            "is_open_access": work["open_access"]["is_oa"]
            
        })
    return extracted_data

extracted_data = extract_relevant_fields(works_data)

print(len(extracted_data))
print(extracted_data[0])

100
{'title': 'Artificial Intelligence in Education: A Review', 'publication_year': 2020, 'cited_by_count': 3963, 'is_open_access': True}


Object-Oriented Programming

To represent individual research publications as Python objects,
I created a base class `ResearchWork`.

A child class `OpenAlexWork` inherits from `ResearchWork`
and adds information about open-access status.

In [40]:
class ResearchWork:
    def __init__(self, title, publication_year, cited_by_count):
        self.title = title
        self.publication_year = publication_year
        self.cited_by_count = cited_by_count

    def summary(self):
        return (
            f"{self.title} ({self.publication_year}) - "
            f"{self.cited_by_count} citations"
        )

In [41]:
class OpenAlexWork(ResearchWork):
    def __init__(
        self,
        title,
        publication_year,
        cited_by_count,
        is_open_access
    ):
        super().__init__(
            title,
            publication_year,
            cited_by_count
        )

        self.is_open_access = is_open_access

In [42]:
first_work = extracted_data[0]

paper = OpenAlexWork(
    first_work["title"],
    first_work["publication_year"],
    first_work["cited_by_count"],
    first_work["is_open_access"]
)

print(paper.title)
print(paper.publication_year)
print(paper.cited_by_count)
print(paper.is_open_access)
print(paper.summary())

Artificial Intelligence in Education: A Review
2020
3963
True
Artificial Intelligence in Education: A Review (2020) - 3963 citations


In [43]:
research_objects = []

for work in extracted_data:
    research_objects.append(
        OpenAlexWork(
            work["title"],
            work["publication_year"],
            work["cited_by_count"],
            work["is_open_access"]
        )
    )

print(len(research_objects))
print(research_objects[0].summary())

100
Artificial Intelligence in Education: A Review (2020) - 3963 citations


In [44]:
#converting to a dataframe

import pandas as pd

df = pd.DataFrame(extracted_data)

print(df.head())
df.info()

                                               title  publication_year  \
0     Artificial Intelligence in Education: A Review              2020   
1  The role of artificial intelligence in achievi...              2020   
2  Revolutionizing healthcare: the role of artifi...              2023   
3  BNAI, NO-TOKEN, and MIND-UNITY: Pillars of a S...              2022   
4  Artificial Intelligence and Management: The Au...              2021   

   cited_by_count  is_open_access  
0            3963            True  
1            3225            True  
2            3130            True  
3            4331            True  
4            2494            True  
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   title             100 non-null    str  
 1   publication_year  100 non-null    int64
 2   cited_by_count    100 non-null    int64
 3   is_open_access   

Validate the data

Before cleaning, inspect the dataset for:

- publication-year range
- missing values
- complete duplicate rows
- duplicate titles
- number of rows and columns

This helps decide what actually needs cleaning instead of applying cleaning steps blindly.

In [45]:
def validate_data(df):
    print("Data Validation")
    print("---------------")

    print(f"Shape: {df.shape}")

    print("\nMissing values per column:")
    print(df.isnull().sum())

    print(f"\nNumber of duplicated rows: {df.duplicated().sum()}")
    print(f"Number of duplicated titles: {df['title'].duplicated().sum()}")

    print(f"Minimum publication year: {df['publication_year'].min()}")
    print(f"Maximum publication year: {df['publication_year'].max()}")

validate_data(df)


Data Validation
---------------
Shape: (100, 4)

Missing values per column:
title               0
publication_year    0
cited_by_count      0
is_open_access      0
dtype: int64

Number of duplicated rows: 0
Number of duplicated titles: 0
Minimum publication year: 2020
Maximum publication year: 2024


In [46]:
# Displaying all duplicate rows in the dataframe
duplicate_rows = df[df.duplicated(keep=False)]

display(duplicate_rows.head(10))

,title,publication_year,cited_by_count,is_open_access


Save validated data

The selected OpenAlex data did not contain missing values or duplicate rows.
After validation, the dataset is saved as CSV for further analysis. (I saved the original API response as JSON. After selecting and validating the fields needed for analysis, I saved the prepared tabular dataset as CSV.)

In [47]:
def save_csv(df, file_path):
    df.to_csv(
        file_path,
        index=False
    )

save_csv(df, "data/cleaned_works.csv")

1. How many publications are there per year?
2. What is the average citation count?
3. How many works are open access vs not open access?

In [48]:
def get_publications_per_year(df):
    return (
        df["publication_year"]
        .value_counts()
        .sort_index()
    )


def get_average_citations(df):
    return df["cited_by_count"].mean()


def get_open_access_counts(df):
    return df["is_open_access"].value_counts()

In [49]:
publications_per_year = get_publications_per_year(df)

average_citations = get_average_citations(df)

open_access_counts = get_open_access_counts(df)

In [50]:
print("Publications per year:")
print(publications_per_year)

print(f"\nAverage citations: {average_citations:.2f}")

print("\nOpen access counts:")
print(open_access_counts)

Publications per year:
publication_year
2020    38
2021    24
2022    16
2023    20
2024     2
Name: count, dtype: int64

Average citations: 1316.05

Open access counts:
is_open_access
True     87
False    13
Name: count, dtype: int64
